# Synthetic Respiratory DISEASES (Hypoxia Mechanisms) + Classification  
**Author:** Sergio Pascual Redondo  
**Date:** 10-02-2026  

## Goal
Build a *tabular* synthetic dataset to classify **hypoxia mechanisms and chronic respiratory patterns** using Machine Learning.

## Labels
`label ∈ {healthy, hypoxia_altitude, hypoxia_anemic, asthma, edema, fibrosis, emphysema}`

## Features (15)
- **Age** (years)
- **HR** (heart rate, bpm)
- **RR** (respiratory rate, breaths/min)

### Gas exchange & ventilation
- **SpO2** (pulse oximetry saturation, %)
- **PaCO2** (arterial CO₂ partial pressure, mmHg)

### Acid–base
- **pH**
- **HCO3** (bicarbonate, mEq/L)

### O₂ transport (anemia vs hypoxemia)
- **Hb** (hemoglobin concentration, g/dL)
- **CaO2** (arterial oxygen content, mL O₂/dL)

### Obstruction / restriction
- **AirwayResistance** (obstruction proxy; asthma/emphysema)
- **FEV1_FVC** (spirometry ratio; obstruction vs restriction patterns)
- **DLCO_percent** (diffusion capacity, % predicted; fibrosis/emphysema)

### Imaging / congestion
- **CXR_score** (chest X-ray severity score, 0–12)
- **B_lines** (lung ultrasound B-lines; interstitial/alveolar involvement)
- **BNP** (natriuretic peptide, pg/mL; congestion proxy, strongest in edema)

> **Note:** The dataset is synthetic but physiology-inspired. We purposely introduce **correlations, noise, and overlap** between classes (no “hard rules”), and we keep the dataset **imbalanced** (more healthy subjects) to mimic real clinical prevalence and motivate the use of **stratified splitting** and **per-class metrics** (macro precision/recall/F1).


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

# Works both from the repository root and from the notebooks directory.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pulmonary_disease.data import generate_pulmonary_synthetic_dataset

SEED = 42
np.random.seed(SEED)

## How the synthetic data are generated (brief)

For each class we sample a latent **severity** variable (`severity ~ Beta(2, 4.5)`) to create realistic variability **within** each label. Then we generate the features using physiology-inspired relationships (plus measurement noise), so classes are **not** separated by “hard rules”.

### Class-specific mechanisms (what we emulate)

- **Healthy:** normal ranges for gas exchange and cardiorespiratory response. The **A–a gradient** is small, and imaging findings are minimal (low `B_lines`, low `CXR_score`).

- **Hypoxia (Altitude):** the primary driver is reduced inspired oxygen due to **lower barometric pressure** (internally sampled `Pbar`).  
  This lowers **alveolar oxygen** (`PAO2`), while the **A–a gradient remains small** (i.e., lungs are not the main problem). Mild hyperventilation is emulated with **lower `PaCO2`**.

- **Hypoxia (Anemic):** oxygenation in the lungs can be relatively preserved (small/normal A–a), but the limiting factor is **low hemoglobin (`Hb`)**, reducing oxygen-carrying capacity. We also add mild **tachycardia (`HR`)** and **tachypnea (`RR`)** tendencies with severity.

- **Hypoxia (Asthma / Obstructive):** modeled as increased **airway obstruction** via higher `AirwayResistance`, with **higher RR/HR**. Gas exchange worsens through V/Q mismatch, reflected by a **moderate A–a gradient** and occasional **CO₂ retention** in severe cases.

- **Hypoxia (Pulmonary edema):** modeled as increased interstitial/alveolar water (latent `lung_water` derived from severity). Imaging evidence increases via **higher `B_lines`** and **higher `CXR_score`**, and oxygenation worsens via a **large A–a gradient** (shunt/VQ effects).

- **Hypoxia (Fibrosis):** modeled as chronic interstitial disease with increased `CXR_score` and **moderately increased `B_lines`** (overlapping with edema but typically less extreme). We emulate a tendency to **hyperventilation** (lower `PaCO2`) and a **moderate-to-high A–a gradient** due to diffusion limitation.

- **Hypoxia (Emphysema):** modeled as chronic obstructive disease with increased `AirwayResistance`, low-to-moderate imaging changes (typically **low `B_lines`**), and a tendency toward **CO₂ retention** (higher `PaCO2`) with partial metabolic compensation (slightly higher `HCO3`). The A–a gradient is **moderate** (V/Q mismatch + loss of effective surface).

### Coherence constraints (physiology-inspired coupling)

- **Alveolar oxygen (`PAO2`)** is generated internally using a simplified alveolar gas relationship:
  - `PiO2 = FiO2 * (Pbar - 47)`
  - `PAO2 = PiO2 - PaCO2 / 0.8`
  (`Pbar` and `PiO2` are used internally but **not stored** to reduce feature redundancy.)

- **Acid–base** is coupled to ventilation:
  - `pH` is computed from `PaCO2` and `HCO3` using a soft physiological relationship + small noise.

- **SpO2** is produced from an internal arterial oxygen (`PaO2`) using a smooth Hb–O₂ dissociation approximation:
  - `PaO2` is generated as `PAO2 - (A–a gradient) + noise`
  - `SpO2` is generated from `PaO2` via a sigmoid-like saturation curve + noise  
  (`PaO2` is computed internally but **not stored**; we keep `SpO2` as the clinical oxygenation feature.)

### Important design choice
We explicitly inject **noise and overlap** (and keep the dataset **imbalanced with more healthy subjects**) so the task cannot be solved by simple if/else rules and requires ML models + robust evaluation (e.g., macro metrics and confusion matrices).


In [ ]:
# Generate the complete dataset from scratch with a fixed seed.
df = generate_pulmonary_synthetic_dataset(seed=SEED)
display(df.head())
print(f"Dataset shape: {df.shape}")

In [ ]:
# Save the generated dataset in the repository's data directory.
data_path = PROJECT_ROOT / "data" / "processed" / "synthetic_pulmonary_dataset.csv"
data_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(data_path, index=False)
print(f"Saved dataset to: {data_path}")

In [ ]:

display(df.head())

display(df.shape)


display(df.isnull().sum())


display(df.describe())


## Preprocessing and exploratory data analysis

The following plots verify class imbalance, plausible feature ranges, correlations, and intentional overlap between diagnostic groups.

In [ ]:
plt.figure(figsize=(10, 5))
df["label"].value_counts().plot(kind="bar")
plt.title("Dataset class distribution")
plt.xlabel("Class")
plt.ylabel("Number of patients")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Updated to your CURRENT features (we do NOT store PaO2, Pbar, PiO2, PAO2)

cols_box = [
    "SpO2",                       # oxygenation
    "PaCO2", "pH", "HCO3",         # ventilation / acid-base
    "RR", "HR",                    # ventilatory + cardiac response
    "Hb", "CaO2",                  # O2 transport (anemia vs hypoxemia)
    "AirwayResistance", "FEV1_FVC",# obstruction markers
    "DLCO_percent",                # diffusion limitation
    "B_lines", "CXR_score",        # imaging / interstitial findings
    "BNP",                         # congestion biomarker
    "Age"                          # demographics
]

for col in cols_box:
    plt.figure(figsize=(15, 10))
    sns.boxplot(data=df, x="label", y=col)
    plt.title(f"Boxplot of {col} by class")

  
    plt.show()


In [ ]:
plt.figure(figsize=(10, 7))
features = df.drop(columns=["label"])
corr = features.corr()


sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("HeatMap ")
plt.tight_layout()
plt.show()


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

columns = ["SpO2", "PaCO2", "Hb", "CaO2", "AirwayResistance", "DLCO_percent", "B_lines", "BNP"]

g = sns.pairplot(df[columns + ["label"]], hue="label")

plt.show()


In [ ]:


df_plot = df.sample(n=min(len(df), 2000), random_state=SEED)

fig = px.scatter(
    df_plot,
    x="PaCO2",
    y="SpO2",
    color="label",
    size="BNP",
    size_max=35,
    opacity=0.7,
    hover_data=["Hb", "CaO2", "B_lines", "CXR_score", "DLCO_percent", "FEV1_FVC", "RR", "HR", "Age"],
    title="SpO2 vs PaCO2 (bubble size = BNP)"
)

fig.show()


## ML: baseline + stronger models

We evaluate several classifiers with:
- train/test split (stratified)
- accuracy + macro-F1
- confusion matrix
- (optional) 5-fold CV macro-F1 for a more stable comparison


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

train_size = 0.70
cv = 3

In [ ]:
scorings = {
    "accuracy": "accuracy",
    "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
}

target = "label"
X = df.drop(target, axis=1)
y = df[target]

In [ ]:

model_lbls = [
    "dt",
    "nb",
    "rf",
]

models = {
    "dt": {
        "name": "Decision Tree",  # nice name to print in results
        "estimator": DecisionTreeClassifier(random_state=SEED),  # the sklearn model
        "param": [{
            "criterion": ["entropy"],            # how to measure split quality
            "max_depth": [*range(1, 20)],        # tree depth (controls overfitting)
            "class_weight": [None, "balanced"],  # helps with imbalanced classes
        }],
    },

    "nb": {
        "name": "Gaussian Naive Bayes",  # nice name to print in results
        "estimator": GaussianNB(),       # Naive Bayes classifier (no random_state)
        "param": [{
            "var_smoothing": [10**exp for exp in range(-3, -13, -1)],  # numeric stability
        }],
    },

    "rf": {
        "name": "Random forest",  # nice name to print in results
        "estimator": RandomForestClassifier(random_state=SEED),  # ensemble of many trees
        "param": [{
            "max_depth": [*range(4, 10)],          # depth of each tree (controls overfitting)
            "n_estimators": [*range(10, 60, 10)],  # number of trees (more = more stable)
        }],
    },
}

In [ ]:
#SPLIT TRAIN AND TEST
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=train_size, random_state=SEED, stratify=y
)
print(f"\nTraining on {len(X_train)} examples")
print(f"Testing on  {len(X_test)} examples")


In [ ]:
# Tune every model against each evaluation objective.
clfs = []
results = pd.DataFrame(
    columns=[
        "scoring",
        "model",
        "best_params",
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]
)

for scoring_name, scorer in scorings.items():
    for model_label in model_lbls:
        clf = GridSearchCV(
            estimator=models[model_label]["estimator"],
            param_grid=models[model_label]["param"],
            cv=cv,
            scoring=scorer,
            n_jobs=1,
            return_train_score=False,
        )
        clf.fit(X_train, y_train)
        clfs.append(clf)

        y_pred = clf.predict(X_test)
        report = classification_report(
            y_test,
            y_pred,
            output_dict=True,
            zero_division=0,
        )

        results.loc[len(results)] = [
            scoring_name,
            models[model_label]["name"],
            clf.best_params_,
            report["accuracy"],
            report["macro avg"]["precision"],
            report["macro avg"]["recall"],
            report["macro avg"]["f1-score"],
        ]

In [ ]:
for score in scorings:
    scoring_filter = score
    display(
        results[results.scoring == scoring_filter]
        .sort_values(by=scoring_filter, ascending=False) #SORT BY THE BEST 
        .style.format(precision=3) #THE NUMBER OF DECIMALS
        .set_caption(f'Results for scoring "{scoring_filter}"')
    )


## Model evaluation and interpretation

For every optimization metric, we inspect the corresponding confusion matrix. We then use permutation importance on the best macro-F1 model to estimate which variables contribute most to out-of-sample performance.

In [ ]:
for score in scorings:
    scoring_filter = score
    best_row = results.loc[results.scoring == scoring_filter, scoring_filter].idxmax(axis=0)

    disp = ConfusionMatrixDisplay.from_estimator(
        estimator=clfs[best_row],
        X=X_test,
        y=y_test,
        xticks_rotation=30
    )
    disp.ax_.set_title(f"Best Model for {score}: {results.at[best_row, 'model']}")
    plt.show()

# Permutation importance for the model with the strongest test macro-F1.
from sklearn.inspection import permutation_importance

best_idx = results["f1_macro"].astype(float).idxmax()
best_model = clfs[best_idx]
importance = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=10,
    random_state=SEED,
    n_jobs=1,
)

importance_df = (
    pd.DataFrame({"feature": X.columns, "importance": importance.importances_mean})
    .sort_values("importance", ascending=True)
    .tail(12)
)

plt.figure(figsize=(9, 6))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.xlabel("Decrease in macro-F1 after permutation")
plt.title("Permutation importance — best macro-F1 model")
plt.tight_layout()
plt.show()


## Reproducibility note

The dataset is regenerated deterministically from `SEED = 42`. Run `python scripts/reproduce.py` from the repository root to execute this notebook in a clean kernel and save the rendered result under `reports/`.